In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("hasil_feature.csv")

feature_cols = [
    "Biaya_Rata_Rata",
    "Domain_Akademik",
    "Domain_Bisnis_Karir",
    "Domain_Olahraga_E-Sport",
    "Domain_Seni_Kreatif",
    "Domain_Teknologi",
    "Domain_Umum_Lainnya",
    "Jenjang_Encoded",
    "Is_Online",
    "Is_Offline"
]

df = df.dropna(subset=feature_cols)

In [ ]:
scaler = MinMaxScaler()
event_features = scaler.fit_transform(df[feature_cols])


In [ ]:
users_df = pd.read_csv("synthetic_users.csv")

N_POS = 5
N_NEG = 5
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

users_df = users_df.dropna(subset=feature_cols + ["Persona_Name"])

# Scaler was fitted on real competition features, then applied to users.
user_features = scaler.transform(users_df[feature_cols])

# Keep raw event rows for label rules. Persona_Name is never fed to model.
events_raw = df.reset_index(drop=True)

matching_rules = {
    "Tech Mahasiswa Online": {
        "positive": lambda events: (events["Domain_Teknologi"] == 1) | (events["Domain_Bisnis_Karir"] == 1),
        "negative": lambda events: (events["Domain_Seni_Kreatif"] == 1) | (events["Domain_Olahraga_E-Sport"] == 1),
    },
    "Kreator Seni SMA Offline": {
        "positive": lambda events: events["Domain_Seni_Kreatif"] == 1,
        "negative": lambda events: (events["Domain_Teknologi"] == 1) | (events["Domain_Bisnis_Karir"] == 1),
    },
    "Atlet SMP/SMA": {
        "positive": lambda events: events["Domain_Olahraga_E-Sport"] == 1,
        "negative": lambda events: (events["Domain_Akademik"] == 1) | (events["Domain_Seni_Kreatif"] == 1),
    },
    "Akademisi Mahasiswa": {
        "positive": lambda events: events["Domain_Akademik"] == 1,
        "negative": lambda events: (events["Domain_Olahraga_E-Sport"] == 1) | (events["Domain_Seni_Kreatif"] == 1),
    },
    "Bisnis/Karir Umum": {
        "positive": lambda events: events["Domain_Bisnis_Karir"] == 1,
        "negative": lambda events: (events["Domain_Seni_Kreatif"] == 1) | (events["Domain_Olahraga_E-Sport"] == 1),
    },
}

X_user = []
X_event = []
y = []

for user_idx, user_row in users_df.reset_index(drop=True).iterrows():
    persona = user_row["Persona_Name"]
    if persona not in matching_rules:
        continue

    rule = matching_rules[persona]
    positive_indices = np.flatnonzero(rule["positive"](events_raw).to_numpy())
    negative_indices = np.flatnonzero(rule["negative"](events_raw).to_numpy())

    if len(positive_indices) == 0 or len(negative_indices) == 0:
        continue

    sampled_pos = rng.choice(positive_indices, size=N_POS, replace=len(positive_indices) < N_POS)
    sampled_neg = rng.choice(negative_indices, size=N_NEG, replace=len(negative_indices) < N_NEG)

    user_vector = user_features[user_idx]

    for event_idx in sampled_pos:
        X_user.append(user_vector)
        X_event.append(event_features[event_idx])
        y.append(1)

    for event_idx in sampled_neg:
        X_user.append(user_vector)
        X_event.append(event_features[event_idx])
        y.append(0)

X_user = np.array(X_user, dtype=np.float32)
X_event = np.array(X_event, dtype=np.float32)
y = np.array(y, dtype=np.float32)

shuffle_idx = rng.permutation(len(y))
X_user = X_user[shuffle_idx]
X_event = X_event[shuffle_idx]
y = y[shuffle_idx]

print(f"Training pairs: {len(y)}")
print(f"Positive pairs: {int(y.sum())}")
print(f"Negative pairs: {int(len(y) - y.sum())}")


In [ ]:
X_user_train, X_user_test, X_event_train, X_event_test, y_train, y_test = train_test_split(
    X_user, X_event, y, test_size=0.2, random_state=42
)

In [ ]:
class CosineSimilarityLayer(tf.keras.layers.Layer):
    def call(self, inputs):
        user_vec, event_vec = inputs
        user_vec = tf.nn.l2_normalize(user_vec, axis=1)
        event_vec = tf.nn.l2_normalize(event_vec, axis=1)
        similarity = tf.reduce_sum(user_vec * event_vec, axis=1, keepdims=True)
        return similarity

In [ ]:
input_dim = len(feature_cols)

user_input = tf.keras.Input(shape=(input_dim,), name="user_preference")
event_input = tf.keras.Input(shape=(input_dim,), name="event_features")

user_tower = tf.keras.layers.Dense(64, activation="relu")(user_input)
user_tower = tf.keras.layers.Dense(32, activation="relu")(user_tower)

event_tower = tf.keras.layers.Dense(64, activation="relu")(event_input)
event_tower = tf.keras.layers.Dense(32, activation="relu")(event_tower)

similarity = CosineSimilarityLayer()([user_tower, event_tower])

output = tf.keras.layers.Dense(1, activation="sigmoid")(similarity)

model = tf.keras.Model(
    inputs=[user_input, event_input],
    outputs=output
)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_preference     │ (None, 10)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ event_features      │ (None, 10)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │        704 │ user_preference[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │        704 │ event_features[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │      2,080 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 32)        │      2,080 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cosine_similarity_… │ (None, 1)         │          0 │ dense_1[0][0],    │
│ (CosineSimilarityL… │                   │            │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 1)         │          2 │ cosine_similarit… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,570 (21.76 KB)

 Trainable params: 5,570 (21.76 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
class StopAtAccuracy(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if logs.get("accuracy") >= 0.95:
            print("Akurasi sudah mencapai target, training dihentikan.")
            self.model.stop_training = True

In [ ]:
history = model.fit(
    [X_user_train, X_event_train],
    y_train,
    validation_data=([X_user_test, X_event_test], y_test),
    epochs=30,
    batch_size=32,
    callbacks=[StopAtAccuracy()]
)

Epoch 1/30
70/75 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9704 - loss: 0.4110Akurasi sudah mencapai target, training dihentikan.
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9823 - loss: 0.4064 - val_accuracy: 0.9882 - val_loss: 0.3992


In [ ]:
user_pref_raw = pd.DataFrame([{
    "Biaya_Rata_Rata": 0,
    "Domain_Akademik": 0,
    "Domain_Bisnis_Karir": 1,
    "Domain_Olahraga_E-Sport": 0,
    "Domain_Seni_Kreatif": 0,
    "Domain_Teknologi": 1,
    "Domain_Umum_Lainnya": 0,
    "Jenjang_Encoded": 4,
    "Is_Online": 1,
    "Is_Offline": 0
}])

user_pref = scaler.transform(user_pref_raw)
user_pref_repeat = np.repeat(user_pref, len(event_features), axis=0)

scores = model.predict([user_pref_repeat, event_features])

df_result = df.copy()
df_result["score"] = scores

top_5 = df_result.sort_values("score", ascending=False).head(5)

top_5[["Judul", "Jenjang", "Penyelenggara", "Biaya_Rata_Rata", "score"]]

47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


,Judul,Jenjang,Penyelenggara,Biaya_Rata_Rata,score
1465,Plannovation: Business Plan Competition 2.0,SMA / Sederajat,Hima Kewirausahaan (CSI) ITBSS,25000,0.534875
1460,ARSTrobot 2025,SMA / Sederajat,Ars University & Teknovasi Robotika Indonesia,150000,0.534875
1455,GESID 2025,Mahasiswa,GESID,0,0.534875
10,The 8th ISCOMBUS,Mahasiswa,Himpunan Mahasiswa Manajemen FEB UMY,50000,0.534875
957,Portal 7 International Competition,SMA / Sederajat,BEM Vokasi IPB,0,0.534875


In [ ]:
model.save('teemo_model.keras')
import pickle
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("✅ Model and scaler saved.")
